# Demo - Train a Tumor/Cyst Classifier on the KiTS23 Data
[KiTS23](https://kits-challenge.org/kits23/) was a competition were teams competed to develop systems for segmentation of kidneys, tumors and cysts.
You can download their dataset [here](https://github.com/neheller/kits23). 

## Feature Handling

### 1. Extract Features
Create a pandas dataframe with the columnes `image_path` and `seg_path`.

In [ ]:
! rv extract \
    --data KITS.csv \
    --output kits_radiomics.parquet \
    --extractor radiomics \
    --label-map KiTS_label_map.json \
    --augment 0

In [ ]:
! rv extract \
    --data KITS.csv \
    --output kits_embeddings.parquet \
    --extractor embeddings \
    --label-map KiTS_label_map.json \
    --augment 10

### 2. Inspect Data

In [ ]:
import pandas as pd

features = pd.read_parquet("kits_radiomics.parquet")

def describe_data(features):
    n_cases = features['case'].nunique()
    n_lesions = len(features[~features['augmented']])
    print(f"Extracted features for {n_lesions} lesions from {n_cases} cases.")

    classes = features['class_id'].unique()
    print(f"Found {len(classes)} classes: {classes}")

    for cl in classes:
        n_cl_lesions = len(features[(features['class_id'] == cl) & (~features['augmented'])])
        print(f"  Class {cl}: {n_cl_lesions} lesions")

    oversampling_factor = (len(features)-n_lesions) / n_lesions 
    print(f"Each lesion was augmented {oversampling_factor:.1f} times on average.")

describe_data(features)

### 3. Split Data
Now that we have features we can split them into a train and a test partition. We also remove all augmentations (if any) from the test partition

In [ ]:

from renal_vision.shared.utils import generate_stratified_group_split

train, test = generate_stratified_group_split(features,group_col="case")
test = test[~test['augmented']].reset_index(drop=True)

train.to_parquet("features_train.parquet", index=False)
test.to_parquet("features_test.parquet", index=False)

print("Train set:")
describe_data(train)
print("\nTest set:")
describe_data(test)

## Training

### 1. [Optional] Generate Task Specifications
Lets specify some meta data that helps us to understand the models output better.

In [ ]:
import json

names_binary = {
    0 : "Tumor",
    1 : "Cyst",
}

with open("names_binary.json","w") as f:
    json.dump(names_binary, f)    

### 2. Train Models

In [ ]:
! rv train \
    --data features_train.parquet \
    --model xgboost \
    --extractor-config features_train.config.json \
    --output-dir binary_model \
    --class-config "names_binary.json"

## Evaluate Models

In [ ]:
# jupyter IPython may throw an error when plotting the confusion matrix.
# If that happens, please run the following command in a terminal instead.
! rv eval \
    --data features_test.parquet \
    --model binary_model/model.pkl \
    --output-dir binary_model